# UD02 · Notebook 2 — Sistemas basados en reglas: diagnóstico guiado con `experta`

**Objetivo**: ver resuelto, de principio a fin, un sistema basado en reglas muy pequeño con `experta` — el mismo ejemplo de diagnóstico de un vehículo de la teoría (§8.5) — antes de construir uno distinto por tu cuenta.

Es un notebook **guiado**: está ya resuelto y explicado. **No se entrega ni se puntúa** — ábrelo para repasar el ejemplo antes de hacer `N04` (sistema de diagnóstico de un PC, con rúbrica) o antes de la prueba escrita del RA2.

Este es el ejemplo más sencillo posible: solo lo justo para que el ciclo reconocer-actuar deje de ser una definición y se convierta en algo que has visto ejecutarse. La UD05 retoma los sistemas expertos con mucho más detalle (marcos, lógica, sistemas híbridos, control inteligente): aquí no hace falta ir más allá.

## Antes de empezar

```bash
pip install "frozendict>=2.3"
pip install --no-deps experta
pip install schema==0.6.7
```

`experta` (1.9.4, de 2019) importa `collections.Mapping`, retirado de Python en la versión 3.10. El parche siguiente va **siempre antes** de `from experta import *` (issue #34 del proyecto), o el import falla con `ImportError: cannot import name 'Mapping' from 'collections'`.

In [ ]:
%pip install "frozendict>=2.3"
%pip install --no-deps experta
%pip install schema==0.6.7

import collections
import collections.abc
if not hasattr(collections, 'Mapping'):
    collections.Mapping = collections.abc.Mapping
    collections.Iterable = collections.abc.Iterable
    collections.MutableMapping = collections.abc.MutableMapping

from experta import *

print("experta importado correctamente")

## Fase 1 — El motor de conocimiento

Un sistema basado en reglas (§8.1) tiene tres piezas: la **memoria de trabajo** (los `Fact` declarados), la **base de conocimiento** (las reglas `@Rule`) y el **motor de inferencia** (la clase `KnowledgeEngine`, que ejecuta el ciclo reconocer-actuar).

- `@DefFacts` declara los hechos iniciales en cuanto se llama a `engine.reset()`.
- `@Rule(...)` escucha unos hechos concretos: en cuanto todos están presentes en la memoria de trabajo, la regla entra en la agenda y se puede disparar.
- `self.declare(Fact(...))` añade un hecho nuevo a la memoria de trabajo — y puede activar otras reglas, que es como se encadena el razonamiento hacia delante (§8.2).
- `salience` es un número de prioridad: a mayor valor, antes se dispara la regla si varias están listas a la vez.

Es el mismo código de la teoría (§8.5), ahora para ejecutarlo tú y ver qué pasa de verdad:

In [ ]:
class DiagnosticoVehiculo(KnowledgeEngine):
    @DefFacts()
    def inicio(self):
        yield Fact(accion="diagnosticar")

    @Rule(Fact(accion="diagnosticar"), salience=10)
    def arrancar(self):
        print("Iniciando diagnóstico del vehículo...")
        self.declare(Fact(bateria="descargada"))
        self.declare(Fact(luces="no_encienden"))

    @Rule(Fact(bateria="descargada"), Fact(luces="no_encienden"))
    def bateria(self):
        self.declare(Fact(causa="bateria_descargada"))

    @Rule(Fact(causa="bateria_descargada"))
    def resultado(self):
        print("CAUSA PROBABLE: Batería descargada")

## Fase 2 — Ejecútalo y mira la memoria de trabajo

`engine.reset()` activa `@DefFacts` (declara el hecho inicial). `engine.run()` ejecuta el ciclo reconocer-actuar hasta que no queda ninguna regla aplicable.

In [ ]:
engine = DiagnosticoVehiculo()
engine.reset()
engine.run()

print("\n---- memoria de trabajo al terminar ----")
print(engine.facts)

Cada línea de `engine.facts` es un hecho declarado, en el orden en que entró en la memoria de trabajo: primero el hecho inicial (`accion`), luego los dos que declara `arrancar` (`bateria`, `luces`) y por último el que declara `bateria` (`causa`). `arrancar` se dispara el primero porque tiene `salience=10`, la prioridad más alta de las tres reglas — si no hubiera ningún hecho con `accion="diagnosticar"`, ninguna regla se habría disparado nunca: así de literal es el ciclo reconocer-actuar.

## Fase 3 — Amplía con una regla más

Añade **una** regla nueva: si `luces="encienden"` y `motor="no_arranca"`, la causa probable es el **motor de arranque**. Para probarla, cambia también `arrancar` para que declare esos dos hechos en vez de los de la batería.

In [ ]:
class DiagnosticoVehiculo2(KnowledgeEngine):
    @DefFacts()
    def inicio(self):
        yield Fact(accion="diagnosticar")

    @Rule(Fact(accion="diagnosticar"), salience=10)
    def arrancar(self):
        print("Iniciando diagnóstico del vehículo...")
        self.declare(Fact(luces="encienden"))
        self.declare(Fact(motor="no_arranca"))

    @Rule(Fact(bateria="descargada"), Fact(luces="no_encienden"))
    def bateria(self):
        self.declare(Fact(causa="bateria_descargada"))

    @Rule(Fact(luces="encienden"), Fact(motor="no_arranca"))
    def motor_arranque(self):
        self.declare(Fact(causa="motor_arranque_averiado"))

    @Rule(Fact(causa="bateria_descargada"))
    def resultado_bateria(self):
        print("CAUSA PROBABLE: Batería descargada")

    @Rule(Fact(causa="motor_arranque_averiado"))
    def resultado_motor(self):
        print("CAUSA PROBABLE: Motor de arranque averiado")

engine2 = DiagnosticoVehiculo2()
engine2.reset()
engine2.run()

Con las luces encendidas, la regla `bateria` ya no puede dispararse (le falta `luces="no_encienden"` en la memoria de trabajo): solo coincide `motor_arranque`, y el diagnóstico cambia sin tocar una sola línea de las reglas anteriores. Eso es lo que hace útil un sistema basado en reglas: cada regla es independiente y el motor decide sola cuál aplica según los hechos presentes.

## Resumen

| Fase | Qué se ha hecho | Dónde está en la teoría |
|---|---|---|
| 1 | Motor de conocimiento con `@DefFacts` y `@Rule`, igual que en la teoría | §8.1, §8.5 |
| 2 | `engine.reset()` + `engine.run()`, y lectura de `engine.facts` | §8.1 (ciclo reconocer-actuar) |
| 3 | Una regla nueva e independiente, con su propio diagnóstico | §8.2 (encadenamiento hacia delante) |

Esto es deliberadamente pequeño: un caso con dos causas posibles y tres reglas. En `N04` vas a construir tú, desde cero, un sistema distinto (diagnóstico de un PC que no arranca) con más reglas y sin este guion. Si quieres entender **de verdad** los sistemas basados en reglas — marcos, redes semánticas, sistemas híbridos con ML, control inteligente — eso llega en la UD05, que dedica una unidad entera a los sistemas expertos.